# Custom NER with spaCy (SPORTS_NAME, ORG_YEAR, WINNING_POSITION)
Steps: 1) load data → 2) convert to `.spacy` → 3) make config → 4) train → 5) load model → 6) test / evaluate

In [ ]:
import json, random
from pathlib import Path

import spacy
from spacy.tokens import DocBin
from tqdm import tqdm

print("spaCy", spacy.__version__)

## 1. Load data

In [ ]:
DATA_DIR = Path("D:/Document/NLP_Project/NER/data")

TRAIN_PATH = DATA_DIR / "train_annotations.json"
VAL_PATH   = DATA_DIR / "validation_annotations.json"   # <-- change to your real validation file name

with open(TRAIN_PATH) as fp:
    train_data = json.load(fp)

if VAL_PATH.exists():
    with open(VAL_PATH) as fp:
        validation_data = json.load(fp)
else:
    # No separate validation file found -> hold out 20% of train data
    print(f"{VAL_PATH.name} not found. Splitting train data 80/20 instead.")
    anns = train_data["annotations"][:]
    random.seed(42)
    random.shuffle(anns)
    cut = int(len(anns) * 0.8)
    train_data = {"classes": train_data["classes"], "annotations": anns[:cut]}
    validation_data = {"classes": train_data["classes"], "annotations": anns[cut:]}

print("train:", len(train_data["annotations"]), "| validation:", len(validation_data["annotations"]))
print("labels:", train_data["classes"])

## 2. Convert to `.spacy` format
> Fix: your old code reused the **same** `DocBin` for both files, so `validation_data.spacy` also contained the training docs.
> Here every file gets its own fresh `DocBin`.

In [ ]:
def convert(data, out_path):
    nlp_blank = spacy.blank("en")
    db = DocBin()
    skipped = 0
    for text, annot in tqdm(data["annotations"]):
        doc = nlp_blank.make_doc(text)
        ents = []
        for start, end, label in annot["entities"]:
            span = doc.char_span(start, end, label=label, alignment_mode="contract")
            if span is None:
                skipped += 1
            else:
                ents.append(span)
        doc.ents = ents
        db.add(doc)
    db.to_disk(out_path)
    print(f"saved {out_path} | skipped entities: {skipped}")

convert(train_data, "./training_data.spacy")
convert(validation_data, "./validation_data.spacy")

## 3. Create config
Your error was: `OSError: [E050] Can't find model 'en_core_web_lg'`.
`--optimize accuracy` makes the config use the `en_core_web_lg` word vectors, and that download failed (corrupt wheel).

**Option A (used here, no download needed):** `--optimize efficiency`. It works well for this kind of templated text.

**Option B:** keep `accuracy`. First fix the download in a terminal with the venv active:
`pip install --no-cache-dir en_core_web_lg` (or download the `.whl` from the spacy-models GitHub releases page and `pip install` that file).

In [ ]:
!python -m spacy init config config.cfg --lang en --pipeline ner --optimize efficiency --force

## 4. Train
Run this **once** (your old notebook ran training twice). Output goes to `./output`; the best model is saved as `output/model-best`.

In [ ]:
!python -m spacy train config.cfg --output ./output --paths.train ./training_data.spacy --paths.dev ./validation_data.spacy

## 5. Load the trained model

In [ ]:
nlp = spacy.load("output/model-best")
print(nlp.pipe_names)
print(nlp.get_pipe("ner").labels)

## 6. Test on new text

In [ ]:
test_texts = [
    "John Smith, your outstanding performance in the Boxing during the 2015 competition has earned you a well-deserved 1st Position. Congratulations!",
    "In recognition of their exceptional performance in the Rowing, Maria Lopez has been awarded the 3rd Position in the 2011 competition.",
]

for text in test_texts:
    doc = nlp(text)
    print(text)
    for ent in doc.ents:
        print(f"   {ent.text!r:25} -> {ent.label_}")
    print()

In [ ]:
# Visual highlight
from spacy import displacy
displacy.render(nlp(test_texts[0]), style="ent", jupyter=True)

In [ ]:
# Reusable function: text -> dict of entities
def extract(text):
    doc = nlp(text)
    result = {}
    for ent in doc.ents:
        result.setdefault(ent.label_, []).append(ent.text)
    return result

extract("Clara Dean, your performance in the Archery during the 2018 competition has earned you a 2nd Position.")

## 7. Evaluate on the validation set (precision / recall / F1)

In [ ]:
!python -m spacy evaluate output/model-best ./validation_data.spacy --output metrics.json

In [ ]:
with open("metrics.json") as f:
    m = json.load(f)

print(f"Precision: {m['ents_p']:.3f} | Recall: {m['ents_r']:.3f} | F1: {m['ents_f']:.3f}\n")
print("Per label:")
for label, s in m["ents_per_type"].items():
    print(f"  {label:18} P={s['p']:.3f}  R={s['r']:.3f}  F={s['f']:.3f}")

## 8. Use the model later (in any other script)
```python
import spacy
nlp = spacy.load("D:/Document/NLP_Project/NER/output/model-best")
doc = nlp("your text here")
print([(e.text, e.label_) for e in doc.ents])
```